# 04 Isolation Forest — SENTINEL
Output for 05: `artifacts/anomaly_detector.joblib` + `*_fe.csv` with `anomaly_score/flag`. Fit on train normal-only.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo = cwd if (cwd / 'src').exists() else next(p for p in cwd.parents if (p / 'src').exists())
sys.path.insert(0, str(repo))
print('repo root:', repo)


In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix
from src.anomaly_detector import AnomalyDetector
from src import config


In [ ]:
train = pd.read_csv(config.ARTIFACTS_DIR / 'train_fe.csv')
valid = pd.read_csv(config.ARTIFACTS_DIR / 'valid_fe.csv')
test = pd.read_csv(config.ARTIFACTS_DIR / 'test_fe.csv')
feats = [c for c in config.ANOMALY_FEATURES if c in train.columns]
print('anomaly feats:', feats)
det = AnomalyDetector().fit(train[feats], train['is_suspicious'])
print('fit on normal-only n=', int((train['is_suspicious']==0).sum()))


In [ ]:
# Score all splits; hist + IF confusion reference (unsupervised, expect low recall)
for name, df in [('train', train), ('valid', valid), ('test', test)]:
    sc = det.score(df[feats])
    df[['anomaly_score','anomaly_flag']] = sc[['anomaly_score','anomaly_flag']]
    print(name, df.groupby('is_suspicious')['anomaly_score'].mean().to_dict())
    print(name, 'confusion\n', confusion_matrix(df['is_suspicious'], df['anomaly_flag']))
train.hist(column='anomaly_score', by='is_suspicious', bins=30);


In [ ]:
det.save(config.ARTIFACTS_DIR / 'anomaly_detector.joblib')
train.to_csv(config.ARTIFACTS_DIR / 'train_fe.csv', index=False)
valid.to_csv(config.ARTIFACTS_DIR / 'valid_fe.csv', index=False)
test.to_csv(config.ARTIFACTS_DIR / 'test_fe.csv', index=False)
print('saved anomaly_detector.joblib + *_fe.csv -> used by 05_graph_construction.ipynb')
